# 02 — itertools, functools, operator

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- maîtriser les combinateurs d'itérables de `itertools`
- utiliser `functools` au-delà de `wraps` : `partial`, `reduce`, `lru_cache`, `singledispatch`
- connaître le module `operator` pour remplacer les lambdas courantes
- combiner ces trois modules pour un code fonctionnel lisible

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- les itérables, itérateurs, générateurs, compréhensions
- les fonctions de première classe, closures, décorateurs
- `functools.wraps` (vu dans le notebook décorateurs)
- le typage (`Callable`, `Iterator`, etc.)

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- asyncio et les itérateurs asynchrones
- NumPy/Pandas (on reste en Python pur ici)

## Plan

1. itertools : itérateurs infinis
2. itertools : combinatoire
3. itertools : filtrage et groupement
4. itertools : chaînage et découpage
5. functools.partial : application partielle
6. functools.reduce : réduction
7. functools.lru_cache : mémoïsation
8. functools.singledispatch : surcharge par type
9. Le module operator
10. Combiner les trois modules
11. Synthèse
12. Exercices

---

## 1. itertools : itérateurs infinis

`itertools` fournit des itérateurs **paresseux** : ils ne calculent la valeur suivante qu'à la demande. Trois produisent des séquences infinies.

In [ ]:
import itertools


### `count(start, step)` : compteur infini

In [ ]:
from itertools import count

compteur = count(10, 2)
print([next(compteur) for _ in range(5)])  # [10, 12, 14, 16, 18]


### `cycle(iterable)` : boucle infinie

In [ ]:
from itertools import cycle, islice

couleurs = cycle(["rouge", "vert", "bleu"])
print(list(islice(couleurs, 7)))


### `repeat(val, n)` : répétition

In [ ]:
from itertools import repeat

list(repeat("X", 5))


---

## 2. itertools : combinatoire

Outils pour générer des combinaisons, permutations, produits cartésiens.

### `product` : produit cartésien

In [ ]:
from itertools import product

list(product("AB", [1, 2]))


In [ ]:
# Équivalent de boucles imbriquées
list(product(range(3), repeat=2))  # paires (i,j) avec i,j in 0..2


### `permutations` et `combinations`

In [ ]:
from itertools import permutations, combinations

print("Permutations:", list(permutations("ABC", 2)))


In [ ]:
print("Combinaisons:", list(combinations("ABC", 2)))


---

## 3. itertools : filtrage et groupement

Filtrer et grouper des éléments paresseusement.

### `filterfalse` : inverse de `filter`

In [ ]:
from itertools import filterfalse

pairs = list(filter(lambda x: x % 2 == 0, range(10)))
impairs = list(filterfalse(lambda x: x % 2 == 0, range(10)))
print(f"Pairs: {pairs}")
print(f"Impairs: {impairs}")


### `takewhile` / `dropwhile`

In [ ]:
from itertools import takewhile, dropwhile

data = [1, 3, 5, 7, 2, 4, 6]
print("takewhile < 6:", list(takewhile(lambda x: x < 6, data)))
print("dropwhile < 6:", list(dropwhile(lambda x: x < 6, data)))


### `groupby` : grouper par clé

In [ ]:
from itertools import groupby

# ATTENTION : groupby suppose que les données sont triées par la clé !
animaux = [
    ("chat", "Mimi"), ("chat", "Felix"),
    ("chien", "Rex"), ("chien", "Buddy"),
    ("poisson", "Nemo"),
]

for espece, groupe in groupby(animaux, key=lambda x: x[0]):
    print(f"{espece}: {[nom for _, nom in groupe]}")


**Piège classique :** `groupby` ne trie pas — il coupe en groupes consécutifs. Si les données ne sont pas triées, il faut les trier avant.

---

## 4. itertools : chaînage et découpage

Combiner et découper des itérables.

### `chain` : concaténer des itérables

In [ ]:
from itertools import chain

list(chain([1, 2], [3, 4], [5, 6]))


### `chain.from_iterable` : concaténer un itérable d'itérables

In [ ]:
listes = [[1, 2], [3, 4], [5, 6]]
list(chain.from_iterable(listes))


### `islice` : découper un itérateur

In [ ]:
from itertools import islice

# islice(iterable, stop) ou islice(iterable, start, stop, step)
gen = (x**2 for x in range(100))
list(islice(gen, 5, 10))


### `batched` (Python 3.12+) : découper en lots

In [ ]:
from itertools import batched

list(batched(range(10), 3))


### `pairwise` (Python 3.10+) : paires glissantes

In [ ]:
from itertools import pairwise

list(pairwise([1, 2, 3, 4, 5]))


### `zip_longest` : zip avec remplissage

In [ ]:
from itertools import zip_longest

list(zip_longest([1, 2, 3], ["a", "b"], fillvalue="?"))


---

## 5. functools.partial : application partielle

`partial` fige certains arguments d'une fonction pour en créer une nouvelle.

In [ ]:
from functools import partial

def puissance(base: int, exposant: int) -> int:
    return base ** exposant

carre = partial(puissance, exposant=2)
cube = partial(puissance, exposant=3)

print(carre(5))  # 25
print(cube(5))   # 125


In [ ]:
# Exemple concret : formater des nombres
from functools import partial

format_euro = partial(format, ".2f")
# Hmm, pas exactement... utilisons une fonction

def formater_prix(montant: float, devise: str = "EUR") -> str:
    return f"{montant:.2f} {devise}"

prix_euro = partial(formater_prix, devise="EUR")
prix_usd = partial(formater_prix, devise="USD")

print(prix_euro(42.5))
print(prix_usd(42.5))


`partial` est préférable à un `lambda` quand on veut un nom et une docstring :

In [ ]:
# lambda (anonyme, pas de docstring)
carre_lambda = lambda x: puissance(x, 2)

# partial (a un __name__, pickle-able)
carre_partial = partial(puissance, exposant=2)

print(carre_partial.func, carre_partial.keywords)


---

## 6. functools.reduce : réduction

`reduce` applique une fonction de 2 arguments cumulativement sur les éléments d'un itérable, de gauche à droite.

In [ ]:
from functools import reduce

# sum([1, 2, 3, 4]) = ((1+2)+3)+4
reduce(lambda acc, x: acc + x, [1, 2, 3, 4])


In [ ]:
# Avec une valeur initiale
reduce(lambda acc, x: acc + x, [1, 2, 3, 4], 10)  # 10 + 1 + 2 + 3 + 4


### Cas d'usage : aplatir une liste de listes

In [ ]:
listes = [[1, 2], [3], [4, 5, 6]]
reduce(lambda acc, lst: acc + lst, listes, [])


**Note :** pour la somme et le produit, préférez `sum()` et `math.prod()`. `reduce` reste utile pour des réductions **non triviales**.

---

## 7. functools.lru_cache : mémoïsation

Met en cache les résultats des appels précédents. LRU = Least Recently Used.

In [ ]:
from functools import lru_cache

@lru_cache(maxsize=256)
def fibonacci(n: int) -> int:
    if n < 2:
        return n
    return fibonacci(n - 1) + fibonacci(n - 2)

fibonacci(100)


In [ ]:
fibonacci.cache_info()


In [ ]:
# Vider le cache
fibonacci.cache_clear()
fibonacci.cache_info()


### `cache` (Python 3.9+) : `lru_cache` sans limite

In [ ]:
from functools import cache

@cache
def factorielle(n: int) -> int:
    if n <= 1:
        return 1
    return n * factorielle(n - 1)

factorielle(20)


**Attention :** les arguments doivent être **hashables** (pas de listes, pas de dicts).

---

## 8. functools.singledispatch : surcharge par type

Permet de créer une fonction générique avec des implémentations spécialisées par type du **premier argument**.

In [ ]:
from functools import singledispatch

@singledispatch
def afficher(val) -> str:
    return f"Valeur générique : {val}"

@afficher.register(int)
def _(val: int) -> str:
    return f"Entier : {val:_}"

@afficher.register(float)
def _(val: float) -> str:
    return f"Flottant : {val:.4f}"

@afficher.register(list)
def _(val: list) -> str:
    return f"Liste de {len(val)} éléments"


In [ ]:
print(afficher(42))
print(afficher(3.14))
print(afficher([1, 2, 3]))
print(afficher("hello"))  # fallback


---

## 9. Le module operator

`operator` fournit des fonctions équivalentes aux opérateurs Python. Plus lisible qu'un lambda, et plus rapide (implémenté en C).

In [ ]:
import operator

# Équivalences
print(operator.add(3, 5))      # 3 + 5
print(operator.mul(4, 7))      # 4 * 7
print(operator.lt(3, 5))       # 3 < 5


### `itemgetter` : accéder par clé/index

In [ ]:
from operator import itemgetter

students = [
    {"nom": "Alice", "note": 15},
    {"nom": "Bob", "note": 18},
    {"nom": "Charlie", "note": 12},
]

# Trier par note (plus lisible que lambda x: x["note"])
sorted(students, key=itemgetter("note"))


### `attrgetter` : accéder par attribut

In [ ]:
from operator import attrgetter
from dataclasses import dataclass

@dataclass
class Etudiant:
    nom: str
    note: float

etudiants = [Etudiant("Alice", 15), Etudiant("Bob", 18), Etudiant("Charlie", 12)]
sorted(etudiants, key=attrgetter("note"))


### `methodcaller` : appeler une méthode

In [ ]:
from operator import methodcaller

mots = ["banane", "Abricot", "cerise"]
sorted(mots, key=methodcaller("lower"))


---

## 10. Combiner les trois modules

La vraie puissance vient de la combinaison.

In [ ]:
from itertools import chain, groupby
from functools import reduce, partial
from operator import itemgetter, add

# Données : ventes par magasin
ventes = [
    {"magasin": "Paris", "montant": 150},
    {"magasin": "Lyon", "montant": 200},
    {"magasin": "Paris", "montant": 300},
    {"magasin": "Lyon", "montant": 100},
    {"magasin": "Paris", "montant": 250},
]

# Total par magasin
ventes_triees = sorted(ventes, key=itemgetter("magasin"))
for mag, group in groupby(ventes_triees, key=itemgetter("magasin")):
    total = reduce(add, (v["montant"] for v in group))
    print(f"{mag}: {total}")


In [ ]:
# Aplatir et filtrer
from itertools import chain, filterfalse

data = [[1, -2, 3], [-4, 5], [6, -7, 8, -9]]
positifs = list(filterfalse(lambda x: x < 0, chain.from_iterable(data)))
print(positifs)


---

## Synthèse

| Module | Fonction | Usage |
|--------|----------|-------|
| `itertools` | `chain`, `product`, `groupby` | Combiner/grouper itérables |
| `itertools` | `batched`, `pairwise` | Découper (3.10+/3.12+) |
| `itertools` | `count`, `cycle`, `repeat` | Séquences infinies |
| `functools` | `partial` | Figer des arguments |
| `functools` | `reduce` | Réduction cumulative |
| `functools` | `lru_cache` / `cache` | Mémoïsation |
| `functools` | `singledispatch` | Surcharge par type |
| `operator` | `itemgetter`, `attrgetter` | Accesseurs rapides |
| `operator` | `add`, `mul`, `lt`, ... | Opérateurs comme fonctions |

### Règles à retenir

1. itertools produit des itérateurs **paresseux** : pas de mémoire gaspillée.
2. `groupby` exige des données **triées** par la clé.
3. `partial` > lambda pour la lisibilité et la sérialisation.
4. `lru_cache` exige des arguments **hashables**.
5. `operator.itemgetter` > `lambda x: x[key]` pour la vitesse et la clarté.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Produit cartésien de couleurs et tailles *(facile)*

Utilisez `itertools.product` pour générer toutes les combinaisons couleur/taille à partir de `couleurs = ['rouge', 'bleu', 'vert']` et `tailles = ['S', 'M', 'L', 'XL']`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Itertools_functools_operator", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
from itertools import product

couleurs = ['rouge', 'bleu', 'vert']
tailles = ['S', 'M', 'L', 'XL']

variantes = list(product(couleurs, tailles))
print(f"{len(variantes)} variantes")
for c, t in variantes:
    print(f"  {c} {t}")
```

</details>

### Exercice 2 — Fenêtre glissante de moyenne *(facile)*

Utilisez `itertools.pairwise` pour calculer les différences successives d'une liste de prix : `[100, 102, 99, 105, 103]`.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Itertools_functools_operator", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
from itertools import pairwise

prix = [100, 102, 99, 105, 103]
diffs = [b - a for a, b in pairwise(prix)]
print(f"Différences : {diffs}")
```

</details>

### Exercice 3 — Top N par catégorie avec groupby *(moyen)*

Données : une liste de produits `{'cat': str, 'nom': str, 'prix': float}`. Utilisez `groupby` + `islice` pour afficher les 2 produits les plus chers par catégorie.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Itertools_functools_operator", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
from itertools import groupby, islice
from operator import itemgetter

produits = [
    {"cat": "fruit", "nom": "pomme", "prix": 2.5},
    {"cat": "fruit", "nom": "mangue", "prix": 4.0},
    {"cat": "fruit", "nom": "banane", "prix": 1.5},
    {"cat": "légume", "nom": "asperge", "prix": 5.0},
    {"cat": "légume", "nom": "carotte", "prix": 1.0},
    {"cat": "légume", "nom": "artichaut", "prix": 3.5},
]

produits.sort(key=itemgetter("cat"))

for cat, groupe in groupby(produits, key=itemgetter("cat")):
    top2 = sorted(groupe, key=itemgetter("prix"), reverse=True)[:2]
    print(f"{cat}: {[(p['nom'], p['prix']) for p in top2]}")
```

</details>

### Exercice 4 — Mémoïsation avec expiration *(moyen)*

Utilisez `@lru_cache` pour mémoïser une fonction `get_rate(currency)` qui simule un appel réseau (print + sleep). Vérifiez les cache hits.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="02_Itertools_functools_operator", exercice=4)


<details>
<summary>📖 Voir la correction</summary>

```python
import time
from functools import lru_cache

@lru_cache(maxsize=32)
def get_rate(currency: str) -> float:
    """Simule un appel API."""
    print(f"Fetching rate for {currency}...")
    rates = {"USD": 1.08, "GBP": 0.86, "JPY": 162.5}
    return rates.get(currency, 1.0)

print(get_rate("USD"))  # fetch
print(get_rate("USD"))  # cache hit
print(get_rate("GBP"))  # fetch
print(get_rate.cache_info())
```

</details>

---

## Ressources externes

### Documentation officielle
- [`itertools`](https://docs.python.org/3/library/itertools.html)
- [`functools`](https://docs.python.org/3/library/functools.html)
- [`operator`](https://docs.python.org/3/library/operator.html)
- [itertools recipes](https://docs.python.org/3/library/itertools.html#itertools-recipes)

### Lectures complémentaires
- Fluent Python, ch. 17 « Iterables, Iterators, and Generators »
- more-itertools (PyPI) : bibliothèque tierce qui étend itertools